In [2]:
from local_llm_toolkit.loaders import UniversalLoader
from local_llm_toolkit.embedders import OllamaEmbedder
from local_llm_toolkit.vectorstores import ChromaVectorStore
from local_llm_toolkit.ingesters import FileIngester
from local_llm_toolkit.chunkers import RecursiveChunk
from local_llm_toolkit.pipelines import FilePipeline
from local_llm_toolkit.agents import ChatAgent, OrchestratorAgent, BaseAgent
from local_llm_toolkit.agents.tools import Tool

import gradio as gr
from openai import OpenAI
import os
from dotenv import load_dotenv

### Set up environment variables

In [3]:
load_dotenv(override=True)
openai_api_key = os.getenv("OPENAI_API_KEY")


### Initialization of Pipeline Class Instances for Ingestion

In [5]:
embedder = OllamaEmbedder(model="nomic-embed-text")

# Option A — Manual setup
vectorstore = ChromaVectorStore.ephemeral(embedder, collection_name="TemplateDB")
loader = UniversalLoader()
chunker = RecursiveChunk(max_tokens=800, overlap_tokens=150)
ingester = FileIngester(roots=["./TestDocuments"])

# Option B — One-liner with FilePipeline
# pipeline = FilePipeline.create("./TestDocuments", embedder)
# pipeline.run()

### Document Ingestion

In [ ]:
items = ingester.collect()

for item in items:
    document = loader.load(item)
    chunks = chunker.chunk(document)
    result = vectorstore.upsert(chunks)
    print(f"{item.name}: {result}")

In [ ]:
#Test Vector Store
vectorstore.query(query = "What's my dogs name?", top_k=1)['documents'][0]

['I am 29 years old\nI have dog named Ralph\nMy eyes are green\nI was born in May\nMy favorite color is forest green\nI am an engineer\nI have no hobbies\nMy name is sharko']

### Create Tools

In [ ]:
# Option A — Manual Tool construction
information_lookup = Tool(
    function=lambda query, top_k=1: str(vectorstore.query(query, top_k)["documents"][0]),
    name="information_lookup",
    description="Search the vector database for information if you do not have sufficient information to answer the question.",
    parameters={
        "properties": {
            "query": {"type": "string", "description": "Natural language query to search."},
            "top_k": {"type": "integer", "description": "Number of results to return.", "default": 5, "maximum": 50},
        },
        "required": ["query"],
    },
)

# Option B — Tool.create() decorator
@Tool.create(description="Search the vector database for information.")
def information_lookup(query: str, top_k: int = 1) -> str:
    return str(vectorstore.query(query, top_k)["documents"][0])

### Create Agents

In [ ]:
load_dotenv(override=True)
openai_api_key = os.getenv("OPENAI_API_KEY")

if openai_api_key:
    print("Using OpenAI API Client")
    client = OpenAI(api_key=openai_api_key)
    model = "gpt-4o-mini"
else:
    print("Using Ollama Local LLM Client")
    client = OpenAI(api_key="ollama", base_url="http://localhost:11434/v1")
    model = "ministral-3:8b"

chatbot_prompt = "You are a helpful AI with access to user information. Use the information_lookup tool to answer personal questions about the user."

chatbot = ChatAgent(system_prompt=chatbot_prompt, model=model, client=client, tools=information_lookup, stream=False)

response = chatbot.call("What is my favorite color?")
print(response)

### Gradio UI

In [ ]:
import logging
logging.basicConfig(level="INFO")
def chatwrapper(message, history):
    print(message)
    response = []
    for token in chatbot.chat(message):
        if token:
            response.append(token)
            yield "".join(response)

demo = gr.ChatInterface(fn=chatwrapper, chatbot=gr.Chatbot()).launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


hello good sir!


In [ ]:
# The orchestrator plans a TaskList from a goal.
# The executor works through each task, using only the tools assigned to it.

orchestrator = OrchestratorAgent(client=client, model=model)

executor = BaseAgent(
    system_prompt="You are a helpful assistant. Complete the task given to you using your available tools.",
    client=client,
    model=model,
    tools=[information_lookup],
)

task_list = orchestrator.run("Find out what the user's favorite color and dog's name are.", executor)

for task in task_list.tasks:
    print(f"[{task.status}] {task.description}")
    print(f"  → {task.result}\n")

### Agentic Workflow — OrchestratorAgent